In [1]:
import random
import math
import numpy as np
import torch
from sklearn import datasets as sklearn_datasets
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import clear_output
import torch.nn as nn

In [2]:
class Sampler:
    def __init__(self, device='cpu'):
        self.device = device

    def sample(self, size=5):
        raise NotImplementedError("Subclasses must implement sample()")

class StandardNormalSampler(Sampler):
    def __init__(self, dim=1, device='cpu'):
        super(StandardNormalSampler, self).__init__(device=device)
        self.dim = dim

    def sample(self, batch_size=10):
        return torch.randn(batch_size, self.dim, device=self.device)


class SwissRollSampler(Sampler):
    def __init__(self, dim=2, device='cpu', noise=0.8, scale=7.5):
        super(SwissRollSampler, self).__init__(device=device)
        assert dim == 2
        self.dim = 2
        self.noise = noise
        self.scale = scale

    def sample(self, batch_size=10):
        batch = sklearn_datasets.make_swiss_roll(
            n_samples=batch_size,
            noise=self.noise
        )[0].astype('float32')[:, [0, 2]] / self.scale
        return torch.tensor(batch, device=self.device)


In [3]:
class EOTConfig:
    def __init__(self,
                 eps: float = 0.1,
                 batch_size: int = 2048,
                 device: str ="cpu",
                 K: int = 32,
                 epoch: int = 100,
                 lmc_steps: int = 100,
                 lmc_step_size: float = 0.003,
                 seed: int = 42,
                 grad_clip = 100000.0,
                 max_diff_exp_clip=100,
                 ema_momentum = 0.999
        ):
        self.device = device
        self.eps = eps
        self.batch_size = batch_size
        self.K = K
        self.epoch = epoch
        self.score_clip = 10000000.0
        self.grad_clip = grad_clip
        self.max_diff_exp_clip = max_diff_exp_clip
        self.lmc_steps = lmc_steps
        self.lmc_step_size = lmc_step_size
        self.seed = seed
        self.ema_momentum = ema_momentum

In [4]:
class MLP(nn.Module):
    def __init__(self, din=2, hidden=128, dout=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(din, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, dout),
        )

    def forward(self, x):
        return self.net(x)

In [5]:
class EOTTrainer:

    def __init__(self, config, source_sampler, target_sampler, model_theta, model_phi, name):
        self.experiment_name = name

        self.config = config
        self.source_sampler = source_sampler
        self.target_sampler = target_sampler

        self.f_theta = model_theta
        self.f_phi = model_phi
        import copy
        self.f_theta_ema = copy.deepcopy(self.f_theta).eval()
        for p in self.f_theta_ema.parameters(): p.requires_grad_(False)
        self.current_step = 0

    def ema_update(self, model, ema):
        m = self.config.ema_momentum
        with torch.no_grad():
            for p, pe in zip(model.parameters(), ema.parameters()):
                pe.mul_(m).add_(p, alpha=1-m)


    def compute_loss(self, x, y):
        cfg = self.config
        broad_shape = list(x.shape)
        broad_shape.insert(1, cfg.K)
        z = torch.randn(size=broad_shape, device=cfg.device)

        x_noisy = x[:, None, :] - math.sqrt(cfg.eps(self.current_step)) * z

        fphi_x = self.f_phi(x)
        ftheta_xnoisy = self.f_theta(x_noisy.reshape(-1, *x_noisy.shape[2:])).view(cfg.batch_size, cfg.K) / cfg.eps(self.current_step)
        ftheta_y = self.f_theta(y)

        diff_in_exp = ftheta_xnoisy - fphi_x
        diff_in_exp_truncated = torch.clamp(diff_in_exp, min=None, max=self.config.max_diff_exp_clip)
        exp_term = torch.exp(diff_in_exp_truncated.to(torch.float64))

        ftheta_y_mean = ftheta_y.mean()
        fphi_x_mean = fphi_x.mean()
        Loss_theor = (fphi_x_mean + exp_term.mean()) * cfg.eps(self.current_step) - ftheta_y_mean
        L_main = Loss_theor
        return L_main

    def train_step(self):
        x = self.source_sampler.sample(self.config.batch_size).to(self.config.device)
        y = self.target_sampler.sample(self.config.batch_size).to(self.config.device)
        self.train_theta = True
        self.train_phi = True
        loss = self.compute_loss(x, y)
        self.opt_both.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.f_theta.parameters(), max_norm=self.config.grad_clip)
        torch.nn.utils.clip_grad_norm_(self.f_phi.parameters(), max_norm=self.config.grad_clip)
        self.opt_both.step()
        self.ema_update(self.f_theta, self.f_theta_ema)
        self.current_step += 1

    def train(self, viz_callback=None):
        print(f"Starting training name = {self.experiment_name}")
        print("-" * 60)

        while True:
            L_main = self.train_step()
            viz_callback(self)
            if (self.current_step >= self.config.epoch):
                break

        print("Training complete!")
        return 0


    def score_y_given_x(self, y, x):
        y = y.detach().requires_grad_(True)

        ft = self.f_theta_ema(y).sum()
        (gy,) = torch.autograd.grad(ft, y, retain_graph=False, create_graph=False)
        sc = (gy - (y - x)) / self.config.eps(self.current_step)

        return torch.clamp(sc, -self.config.score_clip, self.config.score_clip)

    def sample_pi_given_x(self, x):
        cfg = self.config
        y = x + 0.0 * math.sqrt(cfg.eps(self.current_step)) * torch.randn(x.shape[0], x.shape[1], device=cfg.device)

        for _ in range(cfg.lmc_steps):
            y = y.detach()
            sc = self.score_y_given_x(y, x)

            with torch.no_grad():
                y = y + cfg.lmc_step_size * sc + \
                    math.sqrt(2 * cfg.lmc_step_size) * torch.randn_like(y)

        return y.detach()

    def sample_marginal(self, n_x=500):
        cfg = self.config
        xs = self.source_sampler.sample(n_x).to(cfg.device)
        y = xs

        for _ in range(cfg.lmc_steps):
            y = y.detach()
            sc = self.score_y_given_x(y, xs)

            with torch.no_grad():
                y = y + cfg.lmc_step_size * sc + \
                    math.sqrt(2 * cfg.lmc_step_size) * torch.randn_like(y)

        return y.detach()


    def save_checkpoint(self, filename):
        checkpoint = {
            'f_theta_state_dict': self.f_theta_ema.state_dict(),
            'f_phi_state_dict': self.f_phi.state_dict(),
            'epoch': self.current_step,
            'eps': self.config.eps(self.current_step),
        }
        torch.save(checkpoint, filename)

# Training

In [6]:
config = EOTConfig(
     eps = lambda step: 1.0,
     batch_size = 256,
     device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
     K = 256,
     epoch = 5000,
     lmc_steps = 1000,
     lmc_step_size = 0.001,
     seed = 42,
     ema_momentum=0.999,
     max_diff_exp_clip=20,
     grad_clip=1e20
)

In [8]:
def visualize_training(trainer):
    if (trainer.current_step % 1000 == 0):
        print(trainer.current_step)

In [9]:
dimension = 2
trainer = EOTTrainer(
    config=config,
    source_sampler=StandardNormalSampler(dim=dimension),
    target_sampler=SwissRollSampler(),
    model_theta=MLP(din=dimension, hidden=256).to(config.device),
    model_phi=MLP(din=dimension, hidden=256).to(config.device),
    name = f"swissroll_test"
)

trainer.opt_both = torch.optim.AdamW(
            list(trainer.f_phi.parameters()) + list(trainer.f_theta.parameters()),
            lr=3e-4,
            weight_decay=1e-4,
            betas = (0.7, 0.8)
        )

In [15]:
trainer.train(viz_callback=lambda t: visualize_training(t))

Starting training name = swissroll_test
------------------------------------------------------------
Training complete!


0

# Plotting

In [ ]:
data = trainer.sample_marginal(n_x=5000).cpu()
plt.scatter(data[:, 0], data[:, 1], alpha=0.1)